# Attaching Data Assets (raw and processed)
- ophys:
    - Stage_0 (1 session): Natural movies
    - Stage_1 (3 sessions): drifting gratings

- coregistration:
    -   ophys_zstack:
        - Ophys functional (4 sessions, 8 planes) to ophys structure (1 3D image 0-400um) mapping
    - hcr_zstack:
        - HCR to ophys strucure mapping
- HCR:
    - processed:
        - cellxgene table
    - cell_types:
        - cell types for each cell (based on ABC atlas)
        - cell typting measures

In [1]:
import sys
sys.path.append('/root/capsule/code/Preprocessing')

import codeocean
import codeocean.data_asset

from codeocean.data_asset import (
        DataAssetParams
        )
from codeocean.data_asset import DataAssetAttachParams
import aind_session
from aind_session.utils.codeocean_utils import get_data_asset_source_dir, get_codeocean_client, search_data_assets, get_data_asset_search_query, get_subject_data_assets
from analysis_utils import get_running_timestamps, get_running_df, get_stimulus_timestamps
from codeocean.data_asset import DataAssetAttachParams
import boto3
import json
import numpy as np
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")

In [2]:
client = get_codeocean_client()

In [3]:
hcr_mouse_ids = [755252,767018,767022,782149,788406,790322]
registered_zstack_date = {755252: '2024-12-19', 767018: '2025-02-13', 767022: '2025-03-06', 782149: '2025-05-01', 788406: '2025-07-18', 790322: '2025-07-10'}
co_assets_dict = {}
    
for mouse_id in hcr_mouse_ids:
    print(mouse_id)
    co_assets_dict[mouse_id] = {}
    co_assets_dict[mouse_id]['ophys'] = {}
    assets = list(get_subject_data_assets(mouse_id))
    assets.sort(key=lambda x: x.name)
    # search_params["query"] = get_data_asset_search_query(mouse_id=mouse_id)
    # from_co = search_data_assets(search_params) + search_data_assets(
    #     {"query": str(mouse_id)}
    # )
    assets_raw = [asset for asset in assets if 'multiplane-ophys' in asset.tags and 'raw' in asset.tags and 'multisession' not in asset.tags]

    for asset_raw in assets_raw:
        s3_dir = get_data_asset_source_dir(asset_raw.id).as_posix()
        s3 = boto3.resource('s3')
        BUCKET_NAME = s3_dir.split('/')[2]
        PREFIX = '/'.join(s3_dir.split('/')[3:]) + '/'
        KEY = PREFIX + 'session.json'
        obj = s3.Object(BUCKET_NAME, KEY)
        data = obj.get()['Body'].read()
        json_string = data.decode('utf-8')
        session_json = json.loads(json_string)
        session_type = session_json['session_type']
        if session_type == 'STAGE_1':
        # if session_type == 'STAGE_0' or session_type == 'STAGE_1':
            asset_processed = [asset for asset in assets if f"{asset_raw.name}_processed" in asset.name][-1]
            co_assets_dict[mouse_id]['ophys'][asset_raw.name] = {'session_type':session_type,'raw':asset_raw,"processed":asset_processed} 

    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    co_assets_dict[mouse_id]['ophys'] = {ophys_asset_name:co_assets_dict[mouse_id]['ophys'][ophys_asset_name] for ophys_asset_name in ophys_asset_names[:3]}### 3 if only look at STAGE_1, 4 if includes STAGE_0
    for asset_name in co_assets_dict[mouse_id]['ophys']:
        co_assets_dict[mouse_id]['ophys'][asset_name]['glm'] = [asset for asset in assets if f"{asset_name}_glm" in asset.name][-1]
    
   
    co_assets_dict[mouse_id]['hcr'] = {}

    co_assets_dict[mouse_id]['hcr']['processed'] = [asset for asset in assets if 'pairwise-unmixing' in asset.name][-1]
    co_assets_dict[mouse_id]['hcr']['cell_types'] = [asset for asset in assets if asset.name.endswith('mapping')][-1]

    co_assets_dict[mouse_id]['coregistration'] = {}
    try:
        co_assets_dict[mouse_id]['coregistration']['ophys_zstack'] = [asset for asset in assets if 'coreg_id_mapping' in asset.name][-1]
    except:
        co_assets_dict[mouse_id]['coregistration']['ophys_zstack'] = [asset for asset in assets if 'coreg-id-mapping' in asset.name][-1]
    co_assets_dict[mouse_id]['coregistration']['hcr_zstack'] = [asset for asset in assets if 'ctl-czstack' in asset.name][-1]
    
    session_key = registered_zstack_date[mouse_id]

    co_assets_dict[mouse_id]['cortical_zstack'] = {}
    co_assets_dict[mouse_id]['cortical_zstack']['registered'] = [asset for asset in assets if 'cortical-zstack-registration' in asset.tags and session_key in asset.name][-1]
    co_assets_dict[mouse_id]['cortical_zstack']['segmented'] = [asset for asset in assets if 'cortical-zstack-segmentation' in asset.tags and session_key in asset.name][-1]




755252


767018


767022
782149


788406
790322


In [8]:
mouse_id = 755252
co_assets_dict[mouse_id]['hcr']['processed'] = client.data_assets.get_data_asset(data_asset_id="49533435-a093-45bf-8e7c-892e04b91c39")

In [9]:
co_assets = list(np.hstack([[a['raw'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['processed'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a['glm'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
[a for a in co_assets_dict[mouse_id]['hcr'].values()]+\
[a for a in co_assets_dict[mouse_id]['cortical_zstack'].values()]+\
[a for a in co_assets_dict[mouse_id]['coregistration'].values()] for mouse_id in hcr_mouse_ids]))

for asset in co_assets:
    print(asset.id)

9d25d58b-252f-44a0-8153-5ea9340b9ec6
8595d519-f2e0-4662-9909-56dd38d215bf
a288168a-e6ab-4ca8-9cba-3c1c0555fffb
b430ab77-908e-42ba-9e84-417209919e89
76800ff8-d57b-459f-bdd2-4898acfd46d3
cfedfde7-d89d-4b48-8987-97df27242c54
19d188c5-0717-4859-b09e-34c366386b40
96b88c6d-2f85-4f79-88c9-fcc93cdcc11c
00f388c3-7230-499d-b555-4976b690917a
49533435-a093-45bf-8e7c-892e04b91c39
9421ea21-6353-4cb7-86df-d19e99e263d5
04acdd8a-de77-41c3-8454-85c949810e66
082d35bd-04b8-4fda-a3c3-fbb997d09c28
2faf6f83-dc6f-4a6d-9d78-651730c9dc9a
c448324c-019c-4c25-8a8e-e20933abc67b
722d12ae-9bf2-48aa-bf86-983b72bb8363
ab18529d-56db-4675-b5df-46e88b69a8e7
b15eb0a6-9c2e-482b-b2e4-9cdc547e4e75
d7047438-166d-4b50-9cb3-62c251bd0e70
31f4642a-9052-4e98-80eb-8350d46471d4
56accc9e-6de9-484a-ae0d-b4573b2ec72e
64e6cee5-b4a7-473b-becb-e58863f8f65d
6c577e4f-d17b-4719-b202-6c218a1fd21e
05313b6f-f32a-4128-b7f1-7147fd97445e
2a9345b5-be94-4025-ba06-27238718b955
45015f35-c993-4cdd-939d-8907cd531dfd
de9f15be-7a40-4d69-bb64-8dc304e4877d
e

In [10]:
hcr_data_info = {}

for mouse_id in hcr_mouse_ids:
    hcr_data_info[mouse_id] = {}
    hcr_data_info[mouse_id]['ophys'] = {}
    hcr_data_info[mouse_id]['hcr'] = {}
    hcr_data_info[mouse_id]['coregistration'] = {}
    hcr_data_info[mouse_id]['cortical_zstack'] = {}


    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    hcr_data_info[mouse_id]['ophys'] = {f"session_{session_ind+1}":co_assets_dict[mouse_id]['ophys'][ophys_asset_name].copy() for session_ind, ophys_asset_name in enumerate(ophys_asset_names)}
    for session in hcr_data_info[mouse_id]['ophys'].keys():
        hcr_data_info[mouse_id]['ophys'][session]['raw'] = hcr_data_info[mouse_id]['ophys'][session]['raw'].name
        hcr_data_info[mouse_id]['ophys'][session]['processed'] = hcr_data_info[mouse_id]['ophys'][session]['processed'].name
        hcr_data_info[mouse_id]['ophys'][session]['glm'] = hcr_data_info[mouse_id]['ophys'][session]['glm'].name

    hcr_data_info[mouse_id]['hcr']['processed'] = co_assets_dict[mouse_id]['hcr']['processed'].name
    hcr_data_info[mouse_id]['hcr']['cell_types'] = co_assets_dict[mouse_id]['hcr']['cell_types'].name
    hcr_data_info[mouse_id]['coregistration']['ophys_zstack'] = co_assets_dict[mouse_id]['coregistration']['ophys_zstack'].name
    hcr_data_info[mouse_id]['coregistration']['hcr_zstack'] = co_assets_dict[mouse_id]['coregistration']['hcr_zstack'].name
    hcr_data_info[mouse_id]['cortical_zstack']['registered'] = co_assets_dict[mouse_id]['cortical_zstack']['registered'].name
    hcr_data_info[mouse_id]['cortical_zstack']['segmented'] = co_assets_dict[mouse_id]['cortical_zstack']['segmented'].name

hcr_data_info
with open('/root/capsule/code/Preprocessing/hcr_data_info.pkl', 'wb') as f:
    pickle.dump(hcr_data_info, f)
